In [1]:
from __future__ import annotations

import asyncio
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Optional
from urllib.parse import urljoin

import aiohttp
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams

/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Config ───────────────────────────────────────────────────────────────────
INDEX_URL          = "https://man7.org/linux/man-pages/dir_all_alphabetic.html"
OUTPUT_JSONL_PATH  = Path("/scratch4/home/akrik/NTILC/data/man/raw_ai.jsonl")
ERRORS_PATH        = Path("/scratch4/home/akrik/NTILC/data/man/raw_errors_ai.json")
MODEL_ID           = "Qwen/Qwen3-30B-A3B-Instruct-2507"

CUDA_VISIBLE_DEVICES  = "5,6"
TENSOR_PARALLEL_SIZE  = 2

MAX_PAGES: Optional[int] = None
REQUEST_TIMEOUT        = 20.0
TEMP                   = 0.0
MAX_NEW_TOKENS         = 2048
MAX_MODEL_LEN          = 194400
HTTP_CONCURRENCY       = 32
INFERENCE_BATCH_SIZE   = 16
RESUME_FROM_DISK       = True

SECTION_1_PATTERN = re.compile(r"\(1\)$")
DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/128 Safari/537.36"
    )
}

# Must be set before vLLM loads
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

In [3]:
# ── Prompts ───────────────────────────────────────────────────────────────────
EXTRACTION_SYSTEM_PROMPT = """You extract structured JSON from a Linux man-page HTML body.
Return strict JSON only. Do not include markdown or explanation.
Use this schema exactly:
{
  \"name\": \"string\",
  \"one_line\": \"string\",
  \"description\": \"string\",
  \"invocation\": \"string\",
  \"options\": [
    {
      \"flags\": [\"string\"],
      \"arg\": \"string\",
      \"description\": \"string\"
    }
  ]
}
If a field is unknown, use empty string/list.
Only include real options present in the HTML.
"""

EXAMPLE_OUTPUT = {
    "name": "ls",
    "one_line": "list directory contents",
    "description": "List information about files.",
    "invocation": "ls [OPTION]... [FILE]...",
    "options": [
        {"flags": ["-a", "--all"], "arg": "", "description": "do not ignore entries starting with ."}
    ],
}


def build_prompt(*, source_url: str, html: str) -> str:
    return (
        f"SYSTEM PROMPT:\n{EXTRACTION_SYSTEM_PROMPT}\n\n"
        f"EXAMPLE OUTPUT:\n{json.dumps(EXAMPLE_OUTPUT, ensure_ascii=False, indent=2)}\n\n"
        f"SOURCE URL:\n{source_url}\n\n"
        "HTML BODY:\n"
        "```html\n"
        f"{html}\n"
        "```\n"
    )

In [4]:
# ── Step 1: Collect links ─────────────────────────────────────────────────────
print("Fetching index ...")
resp = requests.get(INDEX_URL, timeout=30, headers=DEFAULT_HEADERS)
soup = BeautifulSoup(resp.text, "html.parser")

all_links: List[str] = sorted({
    urljoin(INDEX_URL, a["href"])
    for a in soup.find_all("a")
    if a.get_text(strip=True) and a.get("href") and SECTION_1_PATTERN.search(a.get_text(strip=True))
})

if MAX_PAGES:
    all_links = all_links[:MAX_PAGES]

print(f"Found {len(all_links)} section-1 links")
all_links[:5]

Fetching index ...
Found 1576 section-1 links


['https://man7.org/linux/man-pages/man1/AS.1.html',
 'https://man7.org/linux/man-pages/man1/Firecfg.1.html',
 'https://man7.org/linux/man-pages/man1/Firejail.1.html',
 'https://man7.org/linux/man-pages/man1/Firemon.1.html',
 'https://man7.org/linux/man-pages/man1/PCPCompat.1.html']

In [5]:
# ── Step 2: Resume ────────────────────────────────────────────────────────────
def load_done_urls(path: Path) -> set[str]:
    done = set()
    if path.exists():
        with path.open() as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    if "source_url" in obj:
                        done.add(obj["source_url"])
                except json.JSONDecodeError:
                    pass
    return done


done_urls: set[str] = load_done_urls(OUTPUT_JSONL_PATH) if RESUME_FROM_DISK else set()
todo_links = [u for u in all_links if u not in done_urls]
print(f"{len(done_urls)} already done — {len(todo_links)} remaining")

26 already done — 1550 remaining


In [6]:
# ── Step 3: Fetch all HTML concurrently ───────────────────────────────────────
async def fetch_one(
    session: aiohttp.ClientSession,
    url: str,
    semaphore: asyncio.Semaphore,
) -> tuple[str, str | None]:
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)) as r:
                r.raise_for_status()
                return url, await r.text()
        except Exception as exc:
            print(f"[fetch error] {url}: {exc}")
            return url, None


async def fetch_all(urls: List[str]) -> Dict[str, str | None]:
    semaphore = asyncio.Semaphore(HTTP_CONCURRENCY)
    connector = aiohttp.TCPConnector(limit=HTTP_CONCURRENCY, ssl=False)
    async with aiohttp.ClientSession(headers=DEFAULT_HEADERS, connector=connector) as session:
        tasks = [fetch_one(session, url, semaphore) for url in urls]
        results = {}
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Fetching HTML"):
            url, html = await coro
            results[url] = html
    return results


try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

html_map = asyncio.run(fetch_all(todo_links))
print(f"Fetched {sum(v is not None for v in html_map.values())} pages successfully")

Fetching HTML: 100%|██████████| 1550/1550 [00:06<00:00, 255.71it/s]

Fetched 1550 pages successfully


In [7]:
# ── Step 4: Build prompts ─────────────────────────────────────────────────────
pairs: List[tuple[str, str]] = []
fetch_errors: List[dict] = []

for url in todo_links:
    html = html_map.get(url)
    if html is None:
        fetch_errors.append({"url": url, "error": "fetch_failed"})
        continue
    pairs.append((url, build_prompt(source_url=url, html=html)))

print(f"Built {len(pairs)} prompts — {len(fetch_errors)} fetch errors")

Built 1550 prompts — 0 fetch errors


In [8]:
# ── Step 5: Load model with vLLM ─────────────────────────────────────────────
# NOTE: To change CUDA_VISIBLE_DEVICES or max_model_len you must restart the kernel.
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.8,
    enable_chunked_prefill=True,
    max_num_batched_tokens=32768,
    max_model_len=MAX_MODEL_LEN,
)

sampling_params = SamplingParams(
    temperature=TEMP,
    max_tokens=MAX_NEW_TOKENS,
)

INFO 03-04 16:58:30 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 194400, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.8, 'max_num_batched_tokens': 32768, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
[2026-03-04 16:58:30] WARNING _http.py:857: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-04 16:58:30 [model.py:541] Resolved architecture: Qwen3MoeForCausalLM
INFO 03-04 16:58:30 [model.py:1561] Using max model len 194400


2026-03-04 16:58:30,600	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 03-04 16:58:30 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 03-04 16:58:30 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=2209951) INFO 03-04 16:58:31 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=194400, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_p

...

(EngineCore_DP0 pid=2209951) (Worker_TP0 pid=2209966) INFO 03-04 16:58:51 [default_loader.py:291] Loading weights took 10.61 seconds
(EngineCore_DP0 pid=2209951) (Worker_TP0 pid=2209966) INFO 03-04 16:58:51 [gpu_model_runner.py:4130] Model loading took 28.51 GiB memory and 11.515365 seconds
(EngineCore_DP0 pid=2209951) (Worker_TP0 pid=2209966) INFO 03-04 16:59:00 [backends.py:812] Using cache directory: /home/akrik/.cache/vllm/torch_compile_cache/89f96fb717/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2209951) (Worker_TP0 pid=2209966) INFO 03-04 16:59:00 [backends.py:872] Dynamo bytecode transform time: 8.25 s
(EngineCore_DP0 pid=2209951) (Worker_TP0 pid=2209966) INFO 03-04 16:59:04 [backends.py:302] Cache the graph of compile range (1, 32768) for later use
(EngineCore_DP0 pid=2209951) (Worker_TP1 pid=2209968) INFO 03-04 16:59:05 [backends.py:302] Cache the graph of compile range (1, 32768) for later use
(EngineCore_DP0 pid=2209951) (Worker_TP0 pid=2209966) WARNING 03

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:05<00:00,  9.24it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:04<00:00, 11.81it/s]


(EngineCore_DP0 pid=2209951) (EngineCore_DP0 pid=2209951) (Worker_TP1 pid=2209968) (Worker_TP0 pid=2209966) INFO 03-04 16:59:19 [custom_all_reduce.py:216] Registering 9894 cuda graph addresses
INFO 03-04 16:59:19 [custom_all_reduce.py:216] Registering 9894 cuda graph addresses
(EngineCore_DP0 pid=2209951) (Worker_TP0 pid=2209966) INFO 03-04 16:59:20 [gpu_model_runner.py:5063] Graph capturing finished in 11 secs, took 1.38 GiB
(EngineCore_DP0 pid=2209951) INFO 03-04 16:59:20 [core.py:272] init engine (profile, create kv cache, warmup model) took 28.34 seconds
INFO 03-04 16:59:23 [llm.py:343] Supported tasks: ['generate']


(EngineCore_DP0 pid=2209951) ERROR 03-04 18:56:45 [multiproc_executor.py:246] Worker proc VllmWorker-0 died unexpectedly, shutting down executor.
(EngineCore_DP0 pid=2209951) (Worker_TP1 pid=2209968) WARNING 03-04 18:56:49 [multiproc_executor.py:786] WorkerProc was terminated
(EngineCore_DP0 pid=2209951) ERROR 03-04 18:56:50 [core.py:948] EngineCore encountered a fatal error.
(EngineCore_DP0 pid=2209951) ERROR 03-04 18:56:50 [core.py:948] Traceback (most recent call last):
(EngineCore_DP0 pid=2209951) ERROR 03-04 18:56:50 [core.py:948]   File "/scratch4/home/akrik/base/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 939, in run_engine_core
(EngineCore_DP0 pid=2209951) ERROR 03-04 18:56:50 [core.py:948]     engine_core.run_busy_loop()
(EngineCore_DP0 pid=2209951) ERROR 03-04 18:56:50 [core.py:948]   File "/scratch4/home/akrik/base/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 964, in run_busy_loop
(EngineCore_DP0 pid=2209951) ERROR 03-04 18:56:50 [core.py:948]   

(EngineCore_DP0 pid=2209951) Process EngineCore_DP0:
(EngineCore_DP0 pid=2209951) Traceback (most recent call last):
(EngineCore_DP0 pid=2209951)   File "/home/fbellos/miniconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=2209951)     self.run()
(EngineCore_DP0 pid=2209951)   File "/home/fbellos/miniconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=2209951)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=2209951)   File "/scratch4/home/akrik/base/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 950, in run_engine_core
(EngineCore_DP0 pid=2209951)     raise e
(EngineCore_DP0 pid=2209951)   File "/scratch4/home/akrik/base/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 939, in run_engine_core
(EngineCore_DP0 pid=2209951)     engine_core.run_busy_loop()
(EngineCore_DP0 pid=2209951)   File "/scratch4/home/akrik/base/lib/python3.12/site-packages/vllm/v1/engine/core.py", 

In [9]:
# ── Step 6: Filter prompts that are too long ──────────────────────────────────
# Must run AFTER Step 5 since we need llm.get_tokenizer()
MAX_PROMPT_TOKENS = MAX_MODEL_LEN - MAX_NEW_TOKENS

tokenizer = llm.get_tokenizer()

errors: List[dict] = list(fetch_errors)  # seed with fetch errors
filtered_pairs, skipped = [], []

for url, prompt in tqdm(pairs, desc="Checking token lengths"):
    n = len(tokenizer.encode(prompt))
    if n <= MAX_PROMPT_TOKENS:
        filtered_pairs.append((url, prompt))
    else:
        print(f"[skip] {url} — {n:,} tokens")
        skipped.append({"url": url, "error": f"prompt_too_long_{n}_tokens"})

errors.extend(skipped)
pairs = filtered_pairs
print(f"{len(pairs)} within limit, {len(skipped)} skipped")

Checking token lengths:  19%|█▉        | 293/1550 [00:02<00:08, 153.43it/s]

[skip] https://man7.org/linux/man-pages/man1/g++.1.html — 322,913 tokens


Checking token lengths:  21%|██        | 327/1550 [00:04<00:28, 43.37it/s] 

[skip] https://man7.org/linux/man-pages/man1/gcc.1.html — 323,294 tokens


Checking token lengths: 100%|██████████| 1550/1550 [00:15<00:00, 101.02it/s]

1548 within limit, 2 skipped


In [10]:
# ── Step 7: Inference + save ──────────────────────────────────────────────────
def parse_json_output(raw: str) -> dict:
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if m:
        raw = m.group(0)
    return json.loads(raw)


OUTPUT_JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)
ERRORS_PATH.parent.mkdir(parents=True, exist_ok=True)

total_processed = 0

with OUTPUT_JSONL_PATH.open("a") as out_file:
    for batch_start in tqdm(range(0, len(pairs), INFERENCE_BATCH_SIZE), desc="Inference batches"):
        batch         = pairs[batch_start : batch_start + INFERENCE_BATCH_SIZE]
        urls_batch    = [u for u, _ in batch]
        prompts_batch = [p for _, p in batch]

        t0 = time.time()
        outputs = llm.generate(prompts_batch, sampling_params)
        elapsed = time.time() - t0
        print(f"  batch {batch_start // INFERENCE_BATCH_SIZE}: "
              f"{len(batch)} pages in {elapsed:.1f}s ({elapsed/len(batch):.2f}s/page)")

        for url, output in zip(urls_batch, outputs):
            raw_text = output.outputs[0].text
            try:
                parsed = parse_json_output(raw_text)
                parsed["source_url"] = url
                out_file.write(json.dumps(parsed, ensure_ascii=False) + "\n")
            except Exception as exc:
                errors.append({"url": url, "error": str(exc), "raw": raw_text[:2000]})

        out_file.flush()
        total_processed += len(batch)

with ERRORS_PATH.open("w") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"\nDone. {total_processed} processed, {len(errors)} errors.")
print(f"Results → {OUTPUT_JSONL_PATH}")
print(f"Errors  → {ERRORS_PATH}")

Inference batches:   1%|          | 1/97 [00:29<47:34, 29.73s/it]

  batch 0: 16 pages in 29.7s (1.86s/page)


Inference batches:   2%|▏         | 2/97 [00:55<43:45, 27.64s/it]

  batch 1: 16 pages in 26.2s (1.64s/page)


Inference batches:   3%|▎         | 3/97 [01:12<35:16, 22.51s/it]

...

Inference batches:  99%|█████████▉| 96/97 [26:04<00:16, 16.81s/it]

  batch 95: 16 pages in 15.5s (0.97s/page)


Inference batches: 100%|██████████| 97/97 [26:11<00:00, 16.20s/it]

  batch 96: 12 pages in 6.9s (0.57s/page)

Done. 1548 processed, 204 errors.
Results → /scratch4/home/akrik/NTILC/data/man/raw_ai.jsonl
Errors  → /scratch4/home/akrik/NTILC/data/man/raw_errors_ai.json


[rank1]:[W304 18:56:46.182976138 TCPStore.cpp:125] [c10d] recvValue failed on SocketImpl(fd=201, addr=[localhost]:59736, remote=[localhost]:58897): Failed to recv, got 0 bytes. Connection was likely closed. Did the remote server shutdown or crash?
Exception raised from recvBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:682 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x80 (0x7c204ab7cb80 in /scratch4/home/akrik/base/lib/python3.12/site-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5ffc5b1 (0x7c20251fc5b1 in /scratch4/home/akrik/base/lib/python3.12/site-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5ffd9ad (0x7c20251fd9ad in /scratch4/home/akrik/base/lib/python3.12/site-packages/torch/lib/libtorch_cpu.so)
frame #3: <unknown function> + 0x5ffe55a (0x7c20251fe55a in /scratch4/home/akrik/base/lib/python3.12/site-packages/torch/lib